In [1]:
import pandas as pd
import os
import re
import json
import numpy as np

# --- 1. 設定 -----------------------------------------------------------------
base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression"
# 全ての出力はこのフォルダに保存されます
output_dir = os.path.join(base_path, "20251222_Final_Thesis_Perfect_Pack_v3")
os.makedirs(output_dir, exist_ok=True)

target_datasets = ["base", "base_OH", "base_FP", "all", "all_OH", "all_FP"]
target_vars = ["PCEmax", "Jsc", "Voc", "FF"]
regions = ["Inner", "Outer"]
imputations = ["n", "m"]

THRES_UP = 1.05
EPSILON = 1e-6

# モデル順序と表示パラメータの設定（1種類目用）
model_configs = {
    "PLS":           {"dir": "PLS",           "sub": "20251218_Rebuilt_12Files_PLS_Train_CV10OOF_OOD",        "file": "fixed_20251218_PLS_summary.csv",        "p": ["ncomp"]},
    "svmLinear":     {"dir": "svmLinear",     "sub": "20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD",   "file": "fixed_20251218_SVM_Linear_summary.csv",  "p": ["C"]},
    "gaussprLinear": {"dir": "gaussprLinear", "sub": "20251218_Rebuilt_12Files_GPR_Linear_Train_CV10OOF_OOD",   "file": "fixed_20251218_GPR_Linear_summary.csv",  "p": []},
    "svmRadial":     {"dir": "SVMRadial",     "sub": "20251218_Rebuilt_12Files_SVM_Radial_Train_CV10OOF_OOD",   "file": "fixed_20251218_SVM_Radial_summary.csv",  "p": ["sigma", "C"]},
    "gaussprRadial": {"dir": "gaussPrRadial", "sub": "20251218_Rebuilt_12Files_GPR_Radial_Train_CV10OOF_OOD",   "file": "fixed_20251218_GPR_Radial_summary.csv",  "p": ["sigma"]},
    "gcvEarth":      {"dir": "gcvEarth",      "sub": "20251218_Rebuilt_12Files_gcvEarth_Train_CV10OOF_OOD",   "file": "fixed_20251218_gcvEarth_summary_refilled.csv", "p": ["degree"]},
    "ppr":           {"dir": "ppr",           "sub": "20251218_Rebuilt_12Files_PPR_Train_CV10OOF_OOD",        "file": "fixed_20251218_PPR_summary.csv",        "p": ["nterms"]},
    "knn":           {"dir": "knn",           "sub": "20251218_Rebuilt_12Files_kNN_Train_CV10OOF_OOD",        "file": "fixed_20251218_kNN_summary.csv",        "p": ["k"]},
    "Rborist":       {"dir": "Rborist",       "sub": "20251218_Rebuilt_12Files_RboristLikeRF_Train_CV10OOF_OOD", "file": "fixed_20251218_RboristLikeRF_summary.csv", "p": ["max_features", "min_samples_leaf"]},
    "monmlp":        {"dir": "monmlp",        "sub": "20251218_Rebuilt_12Files_monmlp_Train_CV10OOF_OOD",     "file": "fixed_20251218_monmlp_summary.csv",     "p": ["alpha", "hidden_layer_sizes"]}
}
ordered_models = list(model_configs.keys())

# --- 2. 補助関数（数値フォーマット、パース、データセット名寄せ） -----------
def format_val(v, is_param=True):
    """数値を博論指定のフォーマットに整形（3桁丸め + LaTeX指数表記）"""
    try:
        val = float(v)
        if val == 0: return "0.000"
        abs_v = abs(val)
        # 0.001以上は小数点第3位丸め
        if abs_v >= 0.001:
            return f"{val:.3f}"
        # 0.001未満は $1.25 \times 10^{-4}$ 形式
        else:
            formatted = "{:.2e}".format(val)
            base, exp = formatted.split('e')
            return f"${float(base):.2f} \\times 10^{{{int(exp)}}}$"
    except:
        return str(v)

def get_dataset_key(filename):
    fn = str(filename).lower()
    if "base" in fn:
        if "oh" in fn: return "base_OH"
        if "fp" in fn: return "base_FP"
        return "base"
    if "all" in fn:
        if "oh" in fn: return "all_OH"
        if "fp" in fn: return "all_FP"
        return "all"
    return "unknown"

def parse_param(p_str, p_name):
    if pd.isna(p_str) or p_str == "": return "N/A"
    p_str = str(p_str)
    extracted = "N/A"
    if p_str.startswith("{"):
        try:
            d = json.loads(p_str.replace("'", '"'))
            # キーの読み替え
            key = "min_samples_leaf" if p_name == "min_samples_leaf" else p_name
            if key in d: extracted = str(d[key])
        except: pass
    if extracted == "N/A":
        # monmlpのタプル等を含む正規表現パース
        pattern = rf"{p_name}[^0-9\.]*[=:]?[^0-9\.]*(\([^\)]+\)|[0-9\.e\-]+)"
        match = re.search(pattern, p_str)
        if match: extracted = match.group(1).rstrip(',')
    
    if extracted != "N/A" and not extracted.startswith("("):
        return format_val(extracted)
    return extracted

# --- 3. データロード ---------------------------------------------------------
print("Step 1: Loading all data...")
data_list = []
for mod, cfg in model_configs.items():
    path = os.path.join(base_path, cfg['dir'], cfg['sub'], cfg['file'])
    if os.path.exists(path):
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            d_key = get_dataset_key(row['File'])
            imp = str(row['File'])[0]
            data_list.append({
                'Model': mod, 'Target': row['Target'], 'Dataset': d_key, 'Imp': imp,
                'Inner_R2': row['CV10_R2'], 'Inner_RMSE': row['CV10_RMSE'],
                'Outer_R2': row['OOD_R2'], 'Outer_RMSE': row['OOD_RMSE'],
                'Best_Params': row['Best_Params']
            })
master_df = pd.DataFrame(data_list)

# --- 4. 出力1: Summary (16種) ------------------------------------------------
print("Step 2: Generating Type 1 Summary files (Rounded/Scientific Format)...")
for reg in regions:
    r_label = "Inner" if reg == "Inner" else "Outer"
    for imp in imputations:
        for tgt in target_vars:
            rows = []
            for mod in ordered_models:
                sub = master_df[(master_df['Model'] == mod) & (master_df['Target'] == tgt) & (master_df['Imp'] == imp)]
                if sub.empty: continue
                # スコア行
                score_row = {"Model": mod}
                for ds in target_datasets:
                    m = sub[sub['Dataset'] == ds]
                    if not m.empty:
                        r2_f = format_val(m.iloc[0][f'{r_label}_R2'])
                        rmse_f = format_val(m.iloc[0][f'{r_label}_RMSE'])
                        score_row[ds] = f"{r2_f} ({rmse_f})"
                    else: score_row[ds] = "N/A"
                rows.append(score_row)
                # パラメータ行
                for pn in model_configs[mod]['p']:
                    p_row = {"Model": f"({pn})"}
                    for ds in target_datasets:
                        m = sub[sub['Dataset'] == ds]
                        p_row[ds] = parse_param(m.iloc[0]['Best_Params'], pn) if not m.empty else "N/A"
                    rows.append(p_row)
            pd.DataFrame(rows).to_csv(os.path.join(output_dir, f"Summary_{reg}_{imp}_{tgt}.csv"), index=False)

# --- 5. 出力2: 詳細比率マトリックス (判定2種統合版) ---------------------------
print("Step 3: Generating Type 2 Detailed Matrices (2 Judges included)...")
for metric in ["R2", "RMSE"]:
    final_rows = []
    for mod in ordered_models:
        row_val = {"Model": mod}
        row_j5  = {"Model": f"({mod}_Judge_5pc)"}
        row_jbin = {"Model": f"({mod}_Judge_Binary)"}
        for reg in regions:
            prefix = "CV" if reg == "Inner" else "OOD"
            r_col = f"{reg}_{metric}"
            for tgt in target_vars:
                for ds in target_datasets:
                    col = f"{metric}_Ratio_{prefix}_{tgt}_{ds}"
                    sn = master_df[(master_df['Model']==mod)&(master_df['Target']==tgt)&(master_df['Dataset']==ds)&(master_df['Imp']=='n')]
                    sm = master_df[(master_df['Model']==mod)&(master_df['Target']==tgt)&(master_df['Dataset']==ds)&(master_df['Imp']=='m')]
                    val, j5, jb = "N/A", 0, -1
                    if not sn.empty and not sm.empty:
                        nv, mv = sn.iloc[0][r_col], sm.iloc[0][r_col]
                        if abs(nv) > EPSILON:
                            ratio = mv / nv; val = format_val(ratio)
                            if metric == "R2":
                                j5 = 1 if ratio >= THRES_UP else (-1 if ratio <= 1/THRES_UP else 0)
                                jb = 1 if ratio >= 1.0 else -1
                            else: # RMSE (小さい方が良い)
                                j5 = 1 if ratio <= 0.95 else (-1 if ratio >= 1.05 else 0)
                                jb = 1 if ratio <= 1.0 else -1
                        elif nv <= EPSILON and mv > EPSILON: val, j5, jb = "Inf", 1, 1
                    row_val[col], row_j5[col], row_jbin[col] = val, j5, jb
        final_rows.extend([row_val, row_j5, row_jbin])
    pd.DataFrame(final_rows).to_csv(os.path.join(output_dir, f"Detailed_Ratio_Matrix_{metric}_Unified.csv"), index=False)

# --- 6. 出力3: 多角的統計サマリー --------------------------------------------
print("Step 4: Generating Type 3 Statistical Summaries (Formatted)...")
compare_list = []
for (mod, tgt, ds), group in master_df.groupby(['Model', 'Target', 'Dataset']):
    rn, rm = group[group['Imp'] == 'n'], group[group['Imp'] == 'm']
    if not rn.empty and not rm.empty:
        res = {'Model': mod, 'Target': tgt, 'Dataset': ds}
        for reg in regions:
            for mtr in ["R2", "RMSE"]:
                nv, mv = rn.iloc[0][f'{reg}_{mtr}'], rm.iloc[0][f'{reg}_{mtr}']
                res[f'n_{mtr}_{reg}'], res[f'm_{mtr}_{reg}'] = nv, mv
                if mtr == "R2": res[f'Win_{mtr}_{reg}'] = 1 if mv >= nv - EPSILON else 0
                else: res[f'Win_{mtr}_{reg}'] = 1 if mv <= nv + EPSILON else 0
        compare_list.append(res)
comp_df = pd.DataFrame(compare_list)

for axis in ['Target', 'Model', 'Dataset']:
    summary = comp_df.groupby(axis).agg({
        **{f'{i}_{mtr}_{reg}': 'mean' for i in ['n','m'] for mtr in ['R2','RMSE'] for reg in regions},
        **{f'Win_{mtr}_{reg}': 'sum' for mtr in ['R2','RMSE'] for reg in regions},
        'Model': 'count'
    }).rename(columns={'Model': 'Total_Tests'})
    # 全ての数値列に一括でフォーマット適用
    for c in summary.columns:
        if c != 'Total_Tests': summary[c] = summary[c].apply(lambda x: format_val(x))
    
    # モデル軸の場合のみ指定順にソート
    if axis == 'Model': summary = summary.reindex(ordered_models)
    summary.to_csv(os.path.join(output_dir, f"Summary_Statistics_By_{axis}.csv"))

print(f"=== ALL PROCESS COMPLETE: Final Deliverables are in {output_dir} ===")

Step 1: Loading all data...
Step 2: Generating Type 1 Summary files (Rounded/Scientific Format)...
Step 3: Generating Type 2 Detailed Matrices (2 Judges included)...
Step 4: Generating Type 3 Statistical Summaries (Formatted)...
=== ALL PROCESS COMPLETE: Final Deliverables are in /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/20251222_Final_Thesis_Perfect_Pack_v3 ===


In [2]:
import os
import zipfile

# --- 設定 ---
target_dir = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/20251222_Final_Thesis_Perfect_Pack_v3"
zip_file_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/Final_Thesis_Deliverables.zip"

def zip_csv_files(source_dir, output_zip):
    # ZIPファイルを作成
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        print(f"Starting to zip CSV files in: {source_dir}")
        
        # フォルダ内のファイルを走査
        for root, dirs, files in os.walk(source_dir):
            for file in files:
                # CSVファイルのみを対象にする
                if file.endswith('.csv'):
                    file_path = os.path.join(root, file)
                    # ZIP内でのパス（フォルダ構造を維持せずファイル名のみにする場合は os.path.basename(file_path)）
                    arcname = os.path.relpath(file_path, source_dir)
                    zipf.write(file_path, arcname)
                    print(f"Added: {file}")

    print(f"--- SUCCESS ---")
    print(f"Zip file created at: {output_zip}")

# 実行
if os.path.exists(target_dir):
    zip_csv_files(target_dir, zip_file_path)
else:
    print(f"[ERROR] Target directory not found: {target_dir}")

Starting to zip CSV files in: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/20251222_Final_Thesis_Perfect_Pack_v3
Added: Summary_Inner_m_FF.csv
Added: Summary_Inner_m_Voc.csv
Added: Summary_Outer_n_Jsc.csv
Added: Summary_Statistics_By_Dataset.csv
Added: Summary_Statistics_By_Target.csv
Added: Summary_Outer_n_Voc.csv
Added: Summary_Outer_m_Voc.csv
Added: Summary_Inner_m_PCEmax.csv
Added: Summary_Outer_n_FF.csv
Added: Summary_Inner_n_PCEmax.csv
Added: Summary_Inner_m_Jsc.csv
Added: Summary_Inner_n_FF.csv
Added: Summary_Inner_n_Voc.csv
Added: Summary_Inner_n_Jsc.csv
Added: Summary_Outer_n_PCEmax.csv
Added: Summary_Outer_m_Jsc.csv
Added: Summary_Statistics_By_Model.csv
Added: Summary_Outer_m_PCEmax.csv
Added: Summary_Outer_m_FF.csv
Added: Detailed_Ratio_Matrix_R2_Unified.csv
Added: Detailed_Ratio_Matrix_RMSE_Unified.csv
--- SUCCESS ---
Zip file created at: /Volumes/csbdeep11/_yasu-i/20250303_rebuile